# EV Charging Data — Quality & EDA Report
## Phase 1: Inspect, clean, and explore the charging dataset

**Dataset:** `Data/ev_charging_patterns.csv` (1,320 rows × 20 columns)

This notebook documents data quality, cleaning decisions, derived features, and exploratory analysis.

> **Important:** This dataset has one row per user/session — it does NOT support per-battery degradation or longitudinal SOH claims.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid")

RAW_CSV = Path("../Data/ev_charging_patterns.csv")  # never modified
CLEAN_PQ = Path("data/processed/charging_clean.parquet")
SUMMARY = Path("data/processed/cleaning_summary.json")

## 1. Load raw data and profile

In [ ]:
df_raw = pd.read_csv(RAW_CSV, parse_dates=["Charging Start Time", "Charging End Time"])
print(f"Shape: {df_raw.shape}")
df_raw.dtypes

In [ ]:
df_raw.describe()

In [ ]:
# Null counts
df_raw.isnull().sum()[df_raw.isnull().sum() > 0]

In [ ]:
# Duplicate rows
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

In [ ]:
# Categorical value counts
for col in ["Vehicle Model", "Charger Type", "User Type", "Charging Station Location", "Time of Day", "Day of Week"]:
    print(f"
{col}:")
    print(df_raw[col].value_counts())

## 2. Run cleaning pipeline and load results

In [ ]:
# Run the cleaning script (safe to re-run — idempotent)
import subprocess, sys
result = subprocess.run([sys.executable, "src/prepare_charging_data.py"], capture_output=True, text=True, cwd=".")
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
df = pd.read_parquet(CLEAN_PQ)
summary = json.loads(SUMMARY.read_text())
print(f"Cleaned rows: {len(df)}")
print(f"Flagged rows: {summary['rows_flagged']} / {summary['raw_row_count']} ({summary['rows_flagged']/summary['raw_row_count']*100:.1f}%)")
print(f"Imputed columns: {summary.get('imputed_counts', {})}")

## 3. Missingness heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df_raw.isnull().T, cbar=False, yticklabels=True, cmap="YlOrRd", ax=ax)
ax.set_title("Missing Values per Column (raw CSV)")
ax.set_xlabel("Row index")
plt.tight_layout()
plt.show()

## 4. Duration & energy distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["Charging Duration (hours)"], bins=40, edgecolor="black", alpha=0.7, color="#4C72B0")
axes[0].axvline(df["Charging Duration (hours)"].median(), color="red", ls="--", label=f'Median: {df["Charging Duration (hours)"].median():.2f}h')
axes[0].set(xlabel="Charging Duration (hours)", ylabel="Count", title="Distribution of Charging Duration")
axes[0].legend()

axes[1].hist(df["Energy Consumed (kWh)"].dropna(), bins=40, edgecolor="black", alpha=0.7, color="#55A868")
axes[1].axvline(df["Energy Consumed (kWh)"].median(), color="red", ls="--", label=f'Median: {df["Energy Consumed (kWh)"].median():.1f} kWh')
axes[1].set(xlabel="Energy Consumed (kWh)", ylabel="Count", title="Distribution of Energy Consumed")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Energy by vehicle model & charger type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order_m = df.groupby("Vehicle Model")["Energy Consumed (kWh)"].median().sort_values().index
sns.boxplot(data=df, y="Vehicle Model", x="Energy Consumed (kWh)", order=order_m, hue="Vehicle Model", ax=axes[0], palette="Set2", legend=False)
axes[0].set_title("Energy Consumed by Vehicle Model")

order_c = df.groupby("Charger Type")["Energy Consumed (kWh)"].median().sort_values().index
sns.boxplot(data=df, y="Charger Type", x="Energy Consumed (kWh)", order=order_c, hue="Charger Type", ax=axes[1], palette="Set3", legend=False)
axes[1].set_title("Energy Consumed by Charger Type")
plt.tight_layout()
plt.show()

## 6. Charging rate vs temperature

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(df["Temperature (°C)"], df["Charging Rate (kW)"], c=df["Charging Duration (hours)"], cmap="coolwarm", alpha=0.6, s=15)
fig.colorbar(sc, ax=ax, label="Duration (hours)")
ax.set(xlabel="Temperature (°C)", ylabel="Charging Rate (kW)", title="Charging Rate vs Temperature (colour = duration)")
plt.tight_layout()
plt.show()

## 7. SOC delta vs energy consumed

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df["socDelta"], df["Energy Consumed (kWh)"], alpha=0.4, s=15, color="#C44E52")
neg = df[df["socDelta"] < 0]
if len(neg):
    ax.scatter(neg["socDelta"], neg["Energy Consumed (kWh)"], color="black", s=25, marker="x", label=f"Negative SOC delta ({len(neg)} rows)")
    ax.legend()
ax.set(xlabel="SOC Delta (%)", ylabel="Energy Consumed (kWh)", title="SOC Delta vs Energy Consumed")
plt.tight_layout()
plt.show()

## 8. Time-of-day demand

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hourly = df.groupby("startHour").size()
axes[0].bar(hourly.index, hourly.values, color="#8172B2", edgecolor="black")
axes[0].set(xlabel="Hour of Day", ylabel="Sessions", title="Sessions by Hour")
axes[0].set_xticks(range(0, 24))

tod = df["Time of Day"].value_counts().reindex(["Morning","Afternoon","Evening","Night"], fill_value=0)
axes[1].bar(tod.index, tod.values, color="#CCB974", edgecolor="black")
axes[1].set(xlabel="Time of Day", ylabel="Sessions", title="Sessions by Time-of-Day Category")
plt.tight_layout()
plt.show()

## 9. Data quality flag breakdown

In [ ]:
flags = summary.get("flag_breakdown", {})
if flags:
    fs = dict(sorted(flags.items(), key=lambda x: x[1], reverse=True))
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(list(fs.keys()), list(fs.values()), color="#DD8452", edgecolor="black")
    for b, v in zip(bars, fs.values()):
        ax.text(b.get_width()+5, b.get_y()+b.get_height()/2, str(v), va="center", fontsize=9)
    ax.set(xlabel="Rows Flagged", title="Data Quality Flags")
    plt.tight_layout()
    plt.show()

## 10. Correlation matrix

In [ ]:
num_cols = ["Battery Capacity (kWh)", "Energy Consumed (kWh)", "Charging Duration (hours)",
           "Charging Rate (kW)", "State of Charge (Start %)", "State of Charge (End %)",
           "Temperature (°C)", "Vehicle Age (years)", "socDelta"]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, linewidths=0.5)
ax.set_title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.show()

## 11. Key findings & caveats

1. **96.1% of rows** have at least one data-quality flag — the dataset is synthetically generated with loosely coupled columns.
2. **198 nulls** across 3 columns (Energy, Rate, Distance) — all 66 per column, imputed with vehicle-model group median.
3. **No duplicate rows.** No rows excluded — all kept with flags.
4. **SOC values > 100%** in 23 rows (physically impossible).
5. **268 rows** have starting SOC ≥ ending SOC.
6. **Temperature** is ambient (−10.7 to 73.2 °C), not battery-cell temperature.
7. **One row per user** — no longitudinal battery history. SOH/RUL requires a separate aging dataset.